In [15]:
import pandas as pd
from pathlib import Path
import os
import sys

# força o Spark a usar o MESMO python (.venv) pro worker que roda o driver —
# sem isso, o worker sobe com o python de C:\spark\ (instalação separada,
# sem pyarrow instalado) e mapInPandas quebra com ModuleNotFoundError
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# evita crash silencioso do worker Python no Windows quando numpy/scipy e
# pyarrow carregam runtimes OpenMP conflitantes no mesmo processo — precisa
# ser setado ANTES da SparkSession, pra propagar aos subprocessos worker
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [16]:
# traceback "bonito" do IPython quebra (TokenError) ao formatar erros vindos
# de frames com fonte dinâmica (ex: lambdas de F.filter/F.transform do Spark)
# no Python 3.13 — Plain evita isso e mostra o erro real
%xmode Plain

Exception reporting mode: Plain


In [17]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    # se algum worker crashar de novo, isso imprime o traceback nativo real
    # (segfault etc.) em vez do "worker exited unexpectedly" genérico
    .config("spark.python.worker.faulthandler.enabled", "true")
    .master("local[*]")
    .appName("feature_engineering")
    .getOrCreate()
)

In [18]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [19]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")

df = spark.read.parquet(threat_dataset_path)

In [ ]:
def plot_threat_event(df_threat, show_player_names=False, show_player_positions=False):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal[_zona]
    - defenders_between_ball_goal[_zona]
    - total_players[_zona]
    - atk_def_advantage[_zona]

    Espera um DataFrame Spark contendo exatamente um evento.

    Sempre desenha: linha da bola, linha do meio-campo (x=0) e as
    2 linhas que dividem o campo em 3 terços.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    show_labels = show_player_names or show_player_positions

    def player_label(p):
        parts = []
        if show_player_names:
            parts.append(p["player"]["name"])
        if show_player_positions:
            parts.append(f'({p["position"]["type"]})')
        return " ".join(parts) if parts else None

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in attackers] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_labels else "markers",
        text=[player_label(p) for p in defenders] if show_labels else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)

    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Linhas divisórias fixas: meio-campo + 2 terços
    # ============================

    field_length = right_x - left_x

    zone_boundaries = [
        0.0,                                # meio-campo (half)
        #left_x + field_length / 3,          # início do terço 2
        #left_x + 2 * field_length / 3,      # início do terço 3
    ]

    for x_boundary in zone_boundaries:
        fig.add_vline(
            x=x_boundary,
            line_dash="dot",
            line_width=2,
            line_color="gray"
        )

    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )

    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)

    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Evento: {row['eventTypeDescription']} | "
        f"Posse: {row['eventTeamName']} | "
        f"P. Invertida: {row['flipped_homeTeam']} | "
        f"Ameaça: {row['threat_score']:.3f} | "
        f"Impacto: {row['threat_score_impact']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
    )

    fig.show()

In [21]:
# # exemplo de clearance com posse mantida (sem posse invertida pra pegar ameaça do adversário) seguida de disputa 
# # + 
# # clearance seguida de posse adversária (com posse invertida pra pegar ameaça do adversário)


# events = [
# '9e6a498f54bad910e13edda2bc83167a'
# ]

# for event in events:

#     df_event = df.filter(F.col("eventId") == event)

#     plot_threat_event(df_event, show_player_names=False, show_player_positions=True)

In [22]:
df = df.select(
    'gameId', 
    'competitionId', 
    'competitionName',
    'season', 
    'date',
    'eventId', 
    'period', 
    'eventTypeDescription',
    'startGameClock',
    'startFormattedGameClock',
    'homeTeam',
    'homeTeamName',
    'opponentTeamName',
    'eventTeamName',
    'possession_id',
    'attackingPlayersNorm', 
    'defendingPlayersNorm', 
    'ballsNorm'        
)

df.show(5)

+------+-------------+---------------+---------+----------+--------------------+------+--------------------+--------------+-----------------------+--------+--------------+----------------+-------------+-------------+--------------------+--------------------+--------------------+
|gameId|competitionId|competitionName|   season|      date|             eventId|period|eventTypeDescription|startGameClock|startFormattedGameClock|homeTeam|  homeTeamName|opponentTeamName|eventTeamName|possession_id|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|
+------+-------------+---------------+---------+----------+--------------------+------+--------------------+--------------+-----------------------+--------+--------------+----------------+-------------+-------------+--------------------+--------------------+--------------------+
|  4436|            1| Premier League|2022-2023|2022-08-05|cd74e462bfa72e28d...|     1|             Rebound|            73|                  01:13|   false|Crys

In [23]:
df.count()

448193

In [ ]:
# Achata array<struct> em array<float> de x/y antes do mapInPandas — reduz o
# payload serializado via Arrow (dropa player.id/name/visibility/confidence/
# position, que não entram nos cálculos). GK já sai filtrado aqui pro
# defendingOutfield, usado nas Features 1/2 (casco/stretch); defending_x/y
# e attacking_x/y (com GK) e ball_x/y alimentam as Features 6/7.
# position agora é um struct (type/typeDescription/groupType) — usa .type
# pra comparar com "GK", igual ao "GK" que já vinha da sigla original.
defending_outfield = F.filter("defendingPlayersNorm", lambda p: p["position"]["type"] != "GK")

df = df.withColumns({
    "defending_outfield_x": F.transform(defending_outfield, lambda p: p["x"]),
    "defending_outfield_y": F.transform(defending_outfield, lambda p: p["y"]),
    "defending_x": F.transform("defendingPlayersNorm", lambda p: p["x"]),
    "defending_y": F.transform("defendingPlayersNorm", lambda p: p["y"]),
    "attacking_x": F.transform("attackingPlayersNorm", lambda p: p["x"]),
    "attacking_y": F.transform("attackingPlayersNorm", lambda p: p["y"]),
    "ball_x": F.get("ballsNorm", 0)["x"],
    "ball_y": F.get("ballsNorm", 0)["y"],
}).drop("attackingPlayersNorm", "defendingPlayersNorm", "ballsNorm")

df.printSchema()

In [25]:
import numpy as np

# Colunas temporárias (arrays achatados) criadas só pra alimentar o cálculo
# das features — descartadas do schema de saída do mapInPandas
TEMP_COLS = [
    "defending_outfield_x", "defending_outfield_y",
    "defending_x", "defending_y",
    "attacking_x", "attacking_y",
    "ball_x", "ball_y",
]

# Uma linha por feature: nome da coluna de saída + tipo Spark. Pra adicionar
# uma feature nova: escreve a função dela na célula de baixo, adiciona uma
# linha aqui e uma entrada em FEATURE_FUNCS — o resto (schema, orquestração)
# não muda.
FEATURE_COLUMNS = [
    ("surface_area", DoubleType()),            # Feature 1 — convex hull
    ("stretch_index", DoubleType()),           # Feature 2 — stretch index
    ("numeric_superiority_10m", DoubleType()), # Feature 6 — superioridade 10m
    ("numeric_superiority_20m", DoubleType()), # Feature 7 — superioridade 20m
]

FEATURE_SCHEMA = StructType(
    [f for f in df.schema.fields if f.name not in TEMP_COLS]
    + [StructField(name, dtype) for name, dtype in FEATURE_COLUMNS]
)

In [26]:
from scipy.spatial import ConvexHull, QhullError

# Cada função de feature recebe o mesmo "context": um dict com os arrays
# achatados daquela linha (chaves = nomes das colunas temporárias criadas
# antes do mapInPandas). Retorna um único valor (float ou np.nan).


def _clean_xy(xs, ys):
    """Remove pares (x, y) com NaN — tracking ausente pra aquele jogador."""
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    valid = ~(np.isnan(xs) | np.isnan(ys))
    return xs[valid], ys[valid]


def compute_surface_area(context):
    """Feature 1: área do casco convexo dos defensores de linha (m²)."""
    xs, ys = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) < 3:
        return np.nan
    pts = np.array(sorted(set(zip(xs.tolist(), ys.tolist()))), dtype=float)
    if len(pts) < 3:
        return np.nan
    try:
        area = ConvexHull(pts).volume  # em 2D, .volume = área (.area seria o perímetro)
        return round(float(area), 2)
    except QhullError:
        return np.nan  # pontos colineares -> casco degenerado


def compute_stretch_index(context):
    """Feature 2: distância média dos defensores de linha até o centroide (m)."""
    xs, ys = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) == 0:
        return np.nan
    cx, cy = xs.mean(), ys.mean()
    stretch_index = np.mean(np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2))
    return round(float(stretch_index), 2)


def _count_within_radius(xs, ys, ball_x, ball_y, radius):
    xs, ys = _clean_xy(xs, ys)
    return int(((xs - ball_x) ** 2 + (ys - ball_y) ** 2 <= radius ** 2).sum())


def _numeric_superiority(context, radius):
    bx, by = context["ball_x"], context["ball_y"]
    if pd.isna(bx) or pd.isna(by):
        return np.nan
    defenders = _count_within_radius(context["defending_x"], context["defending_y"], bx, by, radius)
    attackers = _count_within_radius(context["attacking_x"], context["attacking_y"], bx, by, radius)
    return defenders - attackers


def compute_numeric_superiority_10m(context):
    """Feature 6: defensores menos atacantes num raio de 10m da bola."""
    return _numeric_superiority(context, radius=10)


def compute_numeric_superiority_20m(context):
    """Feature 7: defensores menos atacantes num raio de 20m da bola."""
    return _numeric_superiority(context, radius=20)


# nome da coluna (bate com FEATURE_COLUMNS) -> função que calcula ela
FEATURE_FUNCS = {
    "surface_area": compute_surface_area,
    "stretch_index": compute_stretch_index,
    "numeric_superiority_10m": compute_numeric_superiority_10m,
    "numeric_superiority_20m": compute_numeric_superiority_20m,
}

In [27]:
def compute_defensive_features(iterator):
    # roda por lote (batch) de cada partição via Arrow nos executors — nunca
    # materializa o df inteiro no driver, ao contrário de toPandas(). Só
    # orquestra: monta o context de cada linha e chama cada FEATURE_FUNCS —
    # o cálculo em si vive nas funções da célula anterior, então isso não
    # cresce conforme mais features são adicionadas
    for pdf in iterator:
        n = len(pdf)
        results = {name: np.full(n, np.nan) for name, _ in FEATURE_COLUMNS}

        rows = zip(
            pdf["defending_outfield_x"], pdf["defending_outfield_y"],
            pdf["defending_x"], pdf["defending_y"],
            pdf["attacking_x"], pdf["attacking_y"],
            pdf["ball_x"], pdf["ball_y"],
        )
        for i, (out_x, out_y, def_x, def_y, atk_x, atk_y, bx, by) in enumerate(rows):
            context = {
                "defending_outfield_x": out_x, "defending_outfield_y": out_y,
                "defending_x": def_x, "defending_y": def_y,
                "attacking_x": atk_x, "attacking_y": atk_y,
                "ball_x": bx, "ball_y": by,
            }
            for name, func in FEATURE_FUNCS.items():
                results[name][i] = func(context)

        pdf = pdf.drop(columns=TEMP_COLS)
        for name, values in results.items():
            pdf[name] = values

        yield pdf


df_features = df.mapInPandas(compute_defensive_features, schema=FEATURE_SCHEMA)

In [28]:
df_features.select(
    'eventId',
    'surface_area',
    'stretch_index',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
).show(10, truncate=False)

+--------------------------------+------------+-------------+-----------------------+-----------------------+
|eventId                         |surface_area|stretch_index|numeric_superiority_10m|numeric_superiority_20m|
+--------------------------------+------------+-------------+-----------------------+-----------------------+
|cd74e462bfa72e28da37ff657668e24c|1131.61     |19.68        |0.0                    |1.0                    |
|7a57f9447c508eb3c754b1b47e9f7c31|329.36      |10.25        |0.0                    |2.0                    |
|a4036f1e4ecae8a8c0c3ed5a11ea44ea|410.5       |9.0          |2.0                    |2.0                    |
|d4b0ada21119215f2091529df9e20a0b|549.78      |11.45        |1.0                    |3.0                    |
|425847026680ec614ca540e1a0164405|354.15      |8.6          |5.0                    |2.0                    |
|22066d19cec2ec4d055d000224328762|379.44      |9.98         |1.0                    |0.0                    |
|d4a8802cb